[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/03_combined_identifiability_profiles_fim.ipynb)

# Step 03A — Identifiability, effective parameters, and FIM diagnostics

Canonical notebook for the Step 03A reviewer-response identifiability screen.

## 1. Colab/local setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.check_call(command, cwd=cwd)


if _running_in_colab():
    repo_url = "https://github.com/doxav/astromodel_proving.git"
    repo_dir = Path("/content/astromodel_proving")
    if not repo_dir.exists():
        _run(["git", "clone", repo_url, str(repo_dir)])
    os.chdir(repo_dir)
    if (repo_dir / "requirements.txt").exists():
        _run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
else:
    repo_dir = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
    os.chdir(repo_dir)

PROJECT_ROOT = repo_dir.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Interpretation boundary and claim scope

This notebook is a structural-inspection-informed practical identifiability screen.
It is not a full symbolic STRIKE-GOLDD structural identifiability proof.
Profile-loss curves and FIM spectra are local/practical diagnostics around representative centers.
Broad profiles, flat directions, near-zero FIM modes, and raw-parameter confounding are evidence of limited identifiability/sloppiness under Vm-only observation.
They must not be re-labeled as biological degeneracy.
Biological degeneracy requires later accepted-ensemble mechanism decomposition and predictive validation.

Scientific scope at this stage:

- **Targets R1:** distinguish degeneracy from structural non-identifiability, practical non-identifiability, and sloppiness.
- **Targets R4:** clarify effective-parameter interpretation and guard against over-interpreting raw fitted parameters.
- **Partially targets R7:** provide clearer supplemental figures/tables.
- **Does not solve R2/R3/R5/R6:** empirical variability/noise, model-family sensitivity, biological mechanism modes, and predictive robustness are downstream analyses.

## Reuse boundary

This notebook does not parse ATF traces, reconstruct empirical feature thresholds, define accepted ensembles, decompose Kir/gap/leak mechanisms, or perform held-out current prediction. Those are separate downstream steps.

## 3. Configuration

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

from src.step03_identifiability import run_step03_identifiability

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 120)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "identifiability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "representative_conditions": ["CONTROL", "MFA", "BARIUM"],
    "profile_parameter_spaces": ["raw", "effective"],
    "fim_parameter_spaces": ["raw", "effective"],
    "observable_designs": ["sparse", "dense"],
    "claim_scope": "identifiability_screen_not_biological_degeneracy",
}
with open(OUTPUT_DIR / "notebook_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 4. Run Step 03A identifiability analysis

The canonical runner writes all machine-readable Step 03A outputs under `outputs/identifiability/`.

In [ ]:
results = run_step03_identifiability(PROJECT_ROOT)

representative_centers = results["representative_centers"]
effective_map = results["effective_parameter_map"]
invariance = results["invariance_diagnostics"]
profiles = results["profile_likelihoods"]
profile_summary = results["profile_summary"]
fim_spectrum = results["fim_spectrum"]
fim_loadings = results["fim_mode_loadings"]
fim_diagnostics = results["fim_diagnostics"]
observable_benchmark = results["observable_design_benchmark"]
interpretation_notes = results["interpretation_notes"]
reviewer_response = results["reviewer_response_table"]
analysis_summary = results["analysis_summary"]

print("Step 03A outputs:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path.relative_to(PROJECT_ROOT))
print("-", (OUTPUT_DIR / "analysis_summary.json").relative_to(PROJECT_ROOT))
print("-", (OUTPUT_DIR / "notebook_config.json").relative_to(PROJECT_ROOT))

## 5. Representative centers

Representative accepted centers are used only as local diagnostic anchors. They are not accepted ensembles and do not establish biological degeneracy.

In [ ]:
display(representative_centers)

## 6. Effective-parameter map

Raw parameters are grouped by `identifiability_class`. The primary scientific coordinates are effective combinations. Raw `d` and `pk` are not separately interpretable from this reduced Vm-only model when they enter through their product. The effective coordinate `P_gap_eff = d × pk` is the interpretable coordinate.

In [ ]:
effective_map_grouped = effective_map.sort_values(["identifiability_class", "coordinate_type", "parameter"])
display(effective_map_grouped[[
    "identifiability_class",
    "parameter",
    "coordinate_type",
    "effective_parameter",
    "expression",
    "claim_guardrail",
    "reviewer_interpretation",
]])

## 7. Exact `d × pk` invariance diagnostic

Scaling `d` and inversely scaling `pk` preserves `P_gap_eff = d × pk`, `I_kgap`, and the RHS. This is direct evidence of structural raw-parameter confounding in the reduced equations, not evidence of biological degeneracy.

In [ ]:
display(invariance[[
    "condition",
    "current_na",
    "scale_factor",
    "P_gap_eff_a",
    "P_gap_eff_b",
    "I_kgap_a",
    "I_kgap_b",
    "max_abs_rhs_delta",
    "structural_status",
    "claim_scope",
]])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogy(invariance["scale_factor"], invariance["max_abs_rhs_delta"].clip(lower=1e-18), marker="o", label="max |Δ RHS|")
ax.semilogy(invariance["scale_factor"], (invariance["I_kgap_a"] - invariance["I_kgap_b"]).abs().clip(lower=1e-18), marker="s", label="|Δ I_kgap|")
ax.set_xlabel("d scale factor with reciprocal pk scale")
ax.set_ylabel("absolute difference (log scale)")
ax.set_title("Exact d × pk invariance in reduced RHS")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 8. Profile-loss / practical identifiability diagnostics

Terminology guardrail: this is a **profile-loss screen** / **profile-style practical identifiability diagnostic**. It fixes one coordinate at a time and applies an affine Vm nuisance calibration. It is not a definitive profile likelihood because it does not fully re-optimize all nuisance parameters at each fixed value.

In [ ]:
display(profile_summary[[
    "parameter_space",
    "profile_parameter",
    "base_value",
    "min_profile_loss",
    "max_delta_profile_loss",
    "profile_classification",
    "nuisance_refit_method",
    "claim_guardrail",
]])

for space, subspace in profiles.groupby("parameter_space"):
    params = list(subspace["profile_parameter"].drop_duplicates())
    ncols = 2
    nrows = (len(params) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3.2 * nrows), squeeze=False)
    for ax, parameter in zip(axes.ravel(), params):
        sub = subspace[subspace["profile_parameter"] == parameter].sort_values("multiplier")
        ax.plot(sub["multiplier"], sub["delta_profile_loss"], marker="o")
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
        ax.set_xscale("log")
        ax.set_title(f"{space}: {parameter} ({sub['profile_classification'].iloc[0]})")
        ax.set_xlabel("fixed value / center value")
        ax.set_ylabel("Δ profile_loss")
        ax.grid(True, alpha=0.3)
    for ax in axes.ravel()[len(params):]:
        ax.axis("off")
    fig.suptitle(f"Profile-loss screen in {space} parameter space", y=1.02)
    fig.tight_layout()
    plt.show()

## 9. FIM diagnostics and eigenspectra

Raw-space near-zero modes should be interpreted as raw-parameter non-identifiability/sloppiness. Effective-space modes are the primary scientific interpretation coordinates. Both remain local diagnostics around representative centers.

In [ ]:
display(fim_diagnostics[[
    "condition",
    "current_na",
    "parameter_space",
    "fim_is_symmetric",
    "smallest_eigenvalue",
    "largest_eigenvalue",
    "near_zero_mode_count",
    "log10_eigenvalue_span",
    "claim_guardrail",
]])

conditions = list(fim_spectrum["condition"].drop_duplicates())
fig, axes = plt.subplots(1, len(conditions), figsize=(5.5 * len(conditions), 4.2), sharey=True)
if len(conditions) == 1:
    axes = [axes]
for ax, condition in zip(axes, conditions):
    csub = fim_spectrum[fim_spectrum["condition"] == condition]
    for space, sub in csub.groupby("parameter_space"):
        sub = sub.sort_values("mode_index")
        ax.plot(sub["mode_index"], sub["log10_eigenvalue"], marker="o", label=space)
    ax.set_title(condition)
    ax.set_xlabel("mode index")
    ax.grid(True, alpha=0.3)
    ax.legend(title="parameter space")
axes[0].set_ylabel("log10 eigenvalue")
fig.suptitle("FIM eigenspectra: raw vs effective spaces", y=1.03)
fig.tight_layout()
plt.show()

## 10. Dominant stiff/sloppy mode loadings

Dominant loadings identify local stiff/sloppy directions. They should guide identifiability interpretation, not mechanism or phenotype claims.

In [ ]:
dominant_loadings = (
    fim_loadings.sort_values(["condition", "parameter_space", "mode_index", "abs_loading"], ascending=[True, True, True, False])
    .groupby(["condition", "current_na", "parameter_space", "mode_index"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
display(dominant_loadings[[
    "condition",
    "current_na",
    "parameter_space",
    "mode_index",
    "parameter",
    "abs_loading",
    "mode_class",
    "coordinate_type",
    "claim_guardrail",
]].head(40))

## 11. Observable design benchmark

This benchmark documents numerical/provenance sensitivity of the local FIM diagnostic to sparse versus dense observable designs. It does not address empirical variability/noise; that belongs to the threshold-weighted screening notebook.

In [ ]:
display(observable_benchmark)

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(observable_benchmark["observable_design"], observable_benchmark["log10_eigenvalue_span"])
ax.set_ylabel("log10 eigenvalue span")
ax.set_title("Observable-design benchmark for local FIM")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 12. Reviewer-facing interpretation notes

In [ ]:
display(interpretation_notes)
display(reviewer_response)

## 13. Links to next-stage notebooks

### Link to empirical screening stage

The separate notebook `03_threshold_weighted_empirical_screening.ipynb` should be used as Step 03B / 04 to define accepted fits using empirical condition × region × sweep thresholds and reliability-weighted feature contracts.
This identifiability notebook provides parameter-interpretation guardrails; the threshold-screening notebook defines which candidate fits are accepted under experimental uncertainty.

Downstream analyses must handle:

- R2 empirical variability/noise through threshold-weighted accepted-fit screening;
- R3 model assumption sensitivity in model-family/proxy notebooks;
- R5 biological mechanism modes through accepted-ensemble Kir/gap/leak decomposition;
- R6 predictive robustness through held-out current or perturbation validation.

## 14. Output assertions / execution checks

In [ ]:
claim_boundary_text = """
This notebook is not a biological degeneracy result. It is an identifiability screen showing raw/effective parameter limits and local sloppiness under Vm-only observation.
"""

assert (OUTPUT_DIR / "effective_parameter_map.csv").exists()
assert (OUTPUT_DIR / "invariance_diagnostics.csv").exists()
assert (OUTPUT_DIR / "profile_likelihoods.csv").exists() or (OUTPUT_DIR / "profile_loss_screen.csv").exists()
assert (OUTPUT_DIR / "fim_spectrum.csv").exists()
assert (OUTPUT_DIR / "fim_mode_loadings.csv").exists()
assert (OUTPUT_DIR / "fim_diagnostics.csv").exists()
assert (OUTPUT_DIR / "interpretation_notes.csv").exists()
assert (OUTPUT_DIR / "analysis_summary.json").exists()
assert (OUTPUT_DIR / "notebook_config.json").exists()
assert "not" in claim_boundary_text.lower() and "biological degeneracy" in claim_boundary_text.lower()

with open(OUTPUT_DIR / "analysis_summary.json", "r", encoding="utf-8") as f:
    machine_summary = json.load(f)
required_keys = {
    "notebook_name",
    "step_name",
    "claim_scope",
    "critiques_targeted",
    "critiques_not_resolved",
    "representative_centers",
    "n_profile_diagnostics",
    "n_fim_modes",
    "uses_raw_space_fim",
    "uses_effective_space_fim",
    "has_d_pk_invariance_check",
    "has_interpretation_notes",
    "next_required_steps",
}
assert required_keys.issubset(machine_summary)
assert machine_summary["uses_raw_space_fim"] is True
assert machine_summary["uses_effective_space_fim"] is True
assert machine_summary["has_d_pk_invariance_check"] is True
assert machine_summary["has_interpretation_notes"] is True
machine_summary

## Final reviewer-facing conclusion

This notebook supports R1/R4 by showing raw/effective identifiability limits and local sloppiness. It does not claim biological degeneracy. The degeneracy claim must be evaluated downstream using accepted ensembles, mechanism decomposition, and predictive validation.

## Post-execution scientific status

Executed status for reviewer response: Step 03 supports R1/R4/R7 by documenting effective-coordinate guardrails, d-pk invariance, 9 profile diagnostics, and 57 FIM-mode rows. The scientific conclusion remains an identifiability screen: raw/effective sloppiness and non-separability are visible, but biological degeneracy is not claimed here. The required next evidence layers are accepted full-cell ensembles, hidden-mechanism decomposition, held-out prediction, assumption sensitivity, and parameter plausibility.